## Tensor Class

In [ ]:
import math
class Tensor:
    def __init__(self, data, _children=(), op=''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda:None
        self.op = op

    def __repr__(self):
        return f'Tensor(data={self.data}, grad={self.grad})'

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        out = Tensor(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward

        return out

    __radd__ = __add__
    __rmul__ = __mul__

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        return self + (-other) # reuses add and neg

    def __rsub__(self, other): # Reverse Substraction
        other = other if isinstance(other, Tensor) else Tensor(other)

        return other + (-self)

    def __pow__(self, power):
        assert isinstance(power, (int, float))


        out = Tensor(self.data ** power, (self,), f'**{power}')

        def _backward():
            self.grad += (power * (self.data ** (power - 1)) ) * out.grad

        out._backward = _backward

        return out

    def __truediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        return self * (other ** -1)

    def __rtruediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)

        return other * (self ** -1)

    def exp(self):
        out = Tensor(math.exp(self.data), (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward

        return out

    def log(self):
        out = Tensor(math.log(self.data), (self,), 'log')

        def _backward():
            self.grad += (1 / self.data) * out.grad

        out._backward = _backward

        return out

    def abs(self):
        out = Tensor(abs(self.data), (self,), 'abs')

        def _backward():
            self.grad += (1 if self.data>= 0 else -1) * out.grad

        out._backward = _backward

        return out

    def tanh(self):
        out = Tensor(math.tanh(self.data), (self,), 'tanh')

        def _backward():
            self.grad += (1 - out.data ** 2) * out.grad

        out._backward = _backward

        return out

    def zero_grad(self):
        self.grad = 0.0
                
    def backward(self):

        topo = []
    
        visited = set()
    
        def build(v):
    
            if v not in visited:
    
                visited.add(v)
    
                for child in v._prev:
                    build(child)
    
                topo.append(v)
    
        build(self)
    
        self.grad = 1.0
    
        for node in reversed(topo):
            node._backward()

In [3]:
a = Tensor(2.0)
b = Tensor(3.0)

c = a * b
d = c + a

d.backward()

print("d =", d.data)
print("a.grad =", a.grad)
print("b.grad =", b.grad)

d = 8.0
a.grad = 4.0
b.grad = 2.0


In [4]:
x = Tensor(2)

y = ((x + 3) * 4 + 5) / 2

print(y.data)

y.backward()

print(x.grad)

12.5
2.0


In [5]:
x = Tensor(2)

y = ((x.exp() + 1).log()) * x

y.backward()

print(y.data)
print(x.grad)

4.253856022085945
3.8885221669987375


## Activations

### Sigmoid
Formula:

$$ \sigma(x) = \frac{1}{1+e^{-x}} $$

In [6]:
import math
def sigmoid(x):
    out = Tensor(1 / (1 + math.exp(-x.data)), (x,), 'sigmoid')

    def _backward():

        s = out.data
        x.grad += s * (1 - s) * out.grad

    out._backward = _backward

    return out

In [7]:
x = Tensor(0)

y = sigmoid(x)

y.backward()

print(y.data)
print(x.grad)

0.5
0.25


### Tanh
Formula:

$$ \tanh(x) = \frac{e^x-e^{-x}} {e^x+e^{-x}} $$

In [8]:
def tanh(x):
    out = Tensor(math.tanh(x.data), (x,), 'tanh')

    def _backward():
        t = math.tanh(x.data)
        x.grad += (1 - t**2) * out.grad

    out._backward = _backward

    return out

In [9]:
x = Tensor(0)

y = tanh(x)

y.backward()

print(y.data)
print(x.grad)

0.0
1.0


### ReLU
Formula:

$$ ReLU(x) = \max(0,x) $$


In [10]:
def relu(x):
    out = Tensor(max(0, x.data), (x,), 'relu')

    def _backward():
        x.grad += (1.0 if x.data > 0 else 0.0) * out.grad

    out._backward = _backward

    return out

In [11]:
x = Tensor(-5)
y = relu(x)
y.backward()
print(x.grad)

0.0


### Leaky ReLU
Formula:

$$ f(x) = \begin{cases} x & x>0\\ \alpha x & x\le0 \end{cases} $$

In [12]:
def leaky_relu(x, alpha=0.01):
    out = Tensor(x.data if x.data > 0 else alpha * x.data, (x,), 'leaky_rely')

    def _backward():
        x.grad += (1.0 if x.data > 0 else alpha) * out.grad

    out._backward = _backward

    return out

In [13]:
x = Tensor(-5)
y = leaky_relu(x, alpha=0.01)
y.backward()
print(x.grad)
print(y)

0.01
Tensor(data=-0.05, grad=1.0)


### ELU
Formula:

$$ ELU(x)= \begin{cases} x & x>0\\ \alpha(e^x-1) & x\le0 \end{cases} $$
	​


In [14]:
def elu(x, alpha=1.0):
    out = Tensor(x.data if x.data > 0 else alpha * (math.exp(x.data) - 1), (x,), 'elu')

    def _backward():
        x.gard = (1.0 if x.data > 0 else alpha * math.exp(x.data)) * out.grad

    out._backward = _backward

    return out

In [15]:
x = Tensor(-5)
y = elu(x)
y.backward()
print(x.grad)
print(y)

0.0
Tensor(data=-0.9932620530009145, grad=1.0)


### SELU
Formula:

$$ SELU(x) = \lambda \begin{cases} x & x>0\\ \alpha(e^x-1) & x\le0 \end{cases} $$

	​


In [16]:
def selu(x):
    alpha = 1.6732632423543772
    scale = 1.0507009873554805

    out = Tensor(scale * (x.data if x.data > 0 else alpha * (math.exp(x.data) - 1)), (x,), 'selu')

    def _backward():
        x.grad += (scale * (1.0 if x.data > 0 else alpha * math.exp(x.data))) * out.grad

    out._backward = _backward

    return out

In [17]:
x = Tensor(2)
y = relu(x)
z = y * y
z.backward()
print(x.grad)

4.0


## Layers

A neuron computes:

$$ y = w_1x_1 + w_2x_2 + ... + w_nx_n + b $$

then applies an activation:

output=activation(y)

In [21]:
import random

from tensor import Tensor
from activations import relu

class Neuron:
    def __init__(self, nin, activation=relu):
        self.w = [Tensor(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Tensor(0.0)
        self.activation = activation

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w,x)), self.b)
        out = self.activation(act)
        return out

    def parameters(self):
        return self.w + [self.b]

In [24]:
n = Neuron(2)
out = n([2,3])
print(out)


Tensor(data=0.46569817488062215, grad=0.0)


In [25]:
n.parameters()

[Tensor(data=0.22007712894002207, grad=0.0),
 Tensor(data=0.00851463900019267, grad=0.0),
 Tensor(data=0.0, grad=0.0)]

### Linear Class

In [26]:
class Linear:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())

        return params

In [27]:
layer = Linear(3, 4)
x = [1, 2, 3]
y = layer(x)
print(len(layer.parameters()))
print(len(y))

16
4


### MLP

In [37]:
class MLP:
    def __init__(self, nin, layers):
        sizes = [nin] + layers
        self.layer = [Linear(sizes[i], sizes[i+1]) for i in range(len(layers))]

    def __call__(self, x):
        for layer in self.layer:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layer:
            params.extend(layer.parameters())
        return params

In [39]:
model = MLP(2, [4, 4, 1])
x = [2, 3]
out = model(x)
print(out)

[Tensor(data=0.9712706673102762, grad=0.0)]


In [ ]:
nin = 2
layers = [4,4,1]
sizes = [nin] + layers # [2, 4, 4, 1]

layer = [Linear(sizes[i], sizes[i+1]) for i in range(len(layers))]
params = [layer[i].parameters() for i in range(len(layers))]
print(params)


[[Tensor(data=0.9949639253246141, grad=0.0), Tensor(data=-0.37626130890952236, grad=0.0), Tensor(data=0.0, grad=0.0), Tensor(data=-0.8404322074313881, grad=0.0), Tensor(data=-0.17462141262888586, grad=0.0), Tensor(data=0.0, grad=0.0), Tensor(data=0.5344466205529836, grad=0.0), Tensor(data=0.24611886518125559, grad=0.0), Tensor(data=0.0, grad=0.0), Tensor(data=-0.9108986489766384, grad=0.0), Tensor(data=0.5254916800649461, grad=0.0), Tensor(data=0.0, grad=0.0)], [Tensor(data=0.8546343204231162, grad=0.0), Tensor(data=0.8748199354674113, grad=0.0), Tensor(data=0.9124801636601252, grad=0.0), Tensor(data=-0.04216556673418359, grad=0.0), Tensor(data=0.0, grad=0.0), Tensor(data=0.9086074541457847, grad=0.0), Tensor(data=0.12989840904881755, grad=0.0), Tensor(data=-0.6221940067187641, grad=0.0), Tensor(data=-0.8330162190245785, grad=0.0), Tensor(data=0.0, grad=0.0), Tensor(data=-0.5565640916563737, grad=0.0), Tensor(data=-0.46682445654935645, grad=0.0), Tensor(data=0.5470480101166981, grad=0.